# Controlled active-boundary pilot, version 2

**Question:** how does voltage decay into electrically isolated tissue, and how does the fitted decay-to-plateau distance relate to the tissue contributing to each recording?

Version 1 established capture and a planar wave, but showed incomplete adaptation, under-resolved Vm peaks, a conductivity-interface reconstruction error, and insufficient far-region sampling. Version 2 addresses those specific findings. The physical thresholds are unchanged.

The experiment remains a matched intact/lesion pair with full thickness, finite electrodes, fibres along x, and edge pacing. The lesion retains the Courtemanche model with near-insulating conductivities; it is not an ionic model of dead scar.

Read the cells in order. The terminal runner calls the same functions. This revision has been inspected statically but **has not been executed or tested locally**. A successful next run is a hypothesis to verify, not a guaranteed outcome.


## 1. Load the experiment functions

Open this notebook from `3D_Code`, or start Jupyter from the project folder containing `3D_Code` and `Shared_Pipeline`. Use the Python environment that has NumPy, SciPy, pandas and Matplotlib installed. The simulator and mesher must be available separately.


In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

codeFolder = Path.cwd()
if codeFolder.name != "3D_Code":
    codeFolder = codeFolder / "3D_Code"
if not (codeFolder / "boundary_setup.py").is_file():
    raise FileNotFoundError("Start this notebook in 3D_Code or its parent project folder")
sys.path.insert(0, str(codeFolder))

from boundary_setup import (
    loadConfig, validateConfig, prepareExperiment, simulateExperiment,
    readNodePositions, startAnalysis, checkCompletedSimulations, conditionCells,
)
from boundary_geometry import makeElectrodeLayout
from boundary_checks import checkExperiment
from boundary_analysis import (
    calibrateGain, processCondition, computeContributions, saveExperimentResults,
)
from boundary_figures import makeFigures


## 2. Choose the run and experimental settings

The JSON is the single source of defaults. The slab is 40 × 60 × 4 mm at 0.5 mm spacing. Enlarging only y provides room for a 0.5 mm catheter sweep from -16 to +16 mm while retaining 7.5 mm clearance between the outermost electrode samples and the y edges. This is not a domain-convergence claim.

A full pair stores roughly **28 GB of Vm alone**, plus electrode data and derived outputs. Allow at least 40 GB of free disk space and preferably 16 GB RAM. The numerical contribution calculation uses sparse element weights to avoid an element-by-time source array.

Use a new folder for changed simulation settings. For reanalysis, retain the run's saved simulation settings; only analysis, acceptance and clinical settings may differ.


In [ ]:
config = loadConfig(codeFolder / "configs" / "boundary_pilot.json")
runFolder = codeFolder / "runs" / "boundaryPilotV2"
processCount = 1  # Set MPIEXEC_BIN explicitly before using more processes.

# Defaults are kept in one place. Copy the JSON for any scientific sensitivity run.
# V2: 1000 single-cell conditioning beats, then 16 tissue beats at BCL 500 ms.
# Vm/recovery output every 0.2 ms; integration step remains 20 microseconds.
validateConfig(config)
print(json.dumps(config, indent=2))


## 3. Look at the catheter before simulating

The grid has sixteen electrodes. Each electrode is a cylinder aligned with x. We average equally weighted samples on its lateral surface; the default is two axial positions and four angles. `gap` means the distance from the tissue surface to the nearest cylinder surface, not the cylinder centre.

Here we draw the centre-y = 0 placement at the first gap. Other placements translate the same grid across the boundary. No mesh or simulation is created in this step.


In [ ]:
layout = makeElectrodeLayout(config)
placements = layout["placements"]  # columns: centre x, centre y, surface gap (mm)
placementIndex = np.flatnonzero(
    np.isclose(placements[:, 1], 0) & np.isclose(placements[:, 2], config["electrodes"]["gapsMm"][0])
)[0]
centres = layout["centres"][placementIndex]  # (16 electrodes, xyz)
samples = layout["samples"][placementIndex]  # (16 electrodes, surface sample, xyz)

figure, axis = plt.subplots(figsize=(6, 6))
axis.scatter(samples[..., 0], samples[..., 1], s=10, label="Surface samples")
axis.scatter(centres[:, 0], centres[:, 1], marker="x", color="black", label="Centres")
for electrodeIndex, position in enumerate(centres):
    axis.annotate(f"E{electrodeIndex}", position[:2], xytext=(5, 5), textcoords="offset points")
axis.axhline(config["boundary"]["positionMm"], color="black", linestyle="--", label="Lesion boundary")
axis.set(xlabel="x (mm)", ylabel="y (mm)", aspect="equal", title="Grid viewed from above")
axis.legend()
plt.show()
print("Catheter placements:", len(placements))
print("Unique recovery points:", len(layout["recoveryPoints"]))


## 4. Prepare the two openCARP inputs

`prepareExperiment` follows four steps in `boundary_setup.py`:

1. Generate one regular slab using `mesher`, then write intact and lesion element tags on identical nodes.
2. Write the finite-electrode recovery coordinates.
3. Select the full-thickness left-edge stimulus strip. In the lesion condition, exclude every node touching a lesion element.
4. Write a complete parameter file for each condition.

The viable stimulus node sets differ between conditions, but the pacing schedule is identical. All fibres point along x. The function refuses to overwrite an incomplete preparation or reuse incompatible simulation settings.

This cell **runs the mesher**, but does not start openCARP.


In [ ]:
prepareExperiment(runFolder, config)

intactPositions = readNodePositions(runFolder / "mesh" / "intact")
lesionPositions = readNodePositions(runFolder / "mesh" / "lesion")
print("Nodes in each condition:", len(intactPositions))
print("Matching node positions:", np.array_equal(intactPositions, lesionPositions))
print((runFolder / "intact.par").read_text())


## 5. Condition the cell, then run matched tissue experiments

`conditionCells` follows the openCARP bench limit-cycle tutorial: pace one unmodified Courtemanche cell for 1000 beats at 500 ms BCL and save its state immediately before the next stimulus. Both slabs load exactly that file through `imp_region[0].im_sv_init`. `BENCH_BIN` may be used to select the installed bench executable.

Sixteen tissue beats then allow adaptation to propagation and electrotonic coupling. We prespecify beat 16 for measurement and compare beats 14–16. Conditioning does not waive those checks or guarantee a full ionic steady state.

The first call below is explicit so colleagues can inspect the conditioning log and state before starting the slabs. `simulateExperiment` also calls it and reuses the completed state. Incomplete tissue outputs are never silently overwritten.


In [ ]:
conditionCells(runFolder, config)
display(json.loads((runFolder / "conditioning" / "complete.json").read_text()))

simulateExperiment(runFolder, config, processCount=processCount)
checkCompletedSimulations(runFolder)


## 6. Check capture, adaptation and wave propagation

Use the final three beats with half-open cycle windows, excluding the next stimulus endpoint. Limits remain 5% maximum Vm-amplitude change and 5 ms maximum APD90 change, plus the existing capture, response coverage, lesion silence, direction and planarity criteria.

The summary now includes medians, 95th percentiles, failing-node fractions and worst-node IDs. Small exported traces show the worst nodes and a fixed interior reference. A failure is investigated rather than selecting a more favourable beat or relaxing a threshold.

`startAnalysis` archives previous derived outputs and saves the exact current analysis source. It does not alter the raw simulation.


In [ ]:
startAnalysis(runFolder, config)
physiologyPassed = checkExperiment(runFolder, config)
makeFigures(runFolder, config)

display(pd.read_csv(runFolder / "physiologyChecks.csv"))
display(json.loads((runFolder / "physiologyStatus.json").read_text()))
display(Image(filename=str(runFolder / "figures" / "01_activation.png")))

if not physiologyPassed:
    raise RuntimeError("Physiology failed: inspect these outputs before analysing contributions")


## 7. Set one gain from the intact active reference

Use the final beat at the declared intact reference placement. The median amplitude of its nine selected bipolar recordings is scaled to 1.44 mV by default. Keep this **same gain** for every condition, placement and noise repeat.

This is imposed amplitude calibration. It does not independently validate the model's voltage scale.


In [ ]:
# Recheck the gate here so running later cells manually cannot accidentally bypass it.
if not physiologyPassed:
    raise RuntimeError("The active reference must pass the physiological checks")
gain = calibrateGain(runFolder, config)
display(json.loads((runFolder / "results" / "calibration.json").read_text()))


## 8. Build clean and noisy recordings with fixed geometry labels

The existing electrode code still defines all 16 unipoles, 9 selected bipoles and 36 omnipolar cliques. The V2 measurement export retains four declared configurations:

- Unipolar E5.
- Selected bipolar E5–E6 or E5–E9, retaining the actual selected pair.
- Omnipolar E5–E6–E9 and its mirrored triangle E9–E10–E5.

Each geometry is translated independently across the boundary; mirrored triangles are never combined into one profile. The one intact gain is shared by all conditions. Repeat zero is clean; repeats 1–3 add measurement noise, not biological replication.

Distances retain both signs explicitly: `distanceViableMm` is positive on the viable side for the geometric plots; `distanceInsideMm` and clinical `Dist` are positive into the lesion. The known mesh boundary is zero, with no clinical 3 mm offset.


In [ ]:
if not physiologyPassed:
    raise RuntimeError("Inspect the failed physiology before continuing")
measurementTables = []
for condition in ["intact", "lesion"]:
    measurements = processCondition(runFolder, condition, config, gain)
    measurementTables.append(measurements)
    print(condition, "recordings exported:", len(measurements))

display(measurementTables[0][[
    "kind", "label", "noiseRepeat", "ppMv", "distanceViableMm", "selectedElectrodeIds"
]].head(10))


## 9. Estimate tissue contributions using tetrahedra

`Shared_Pipeline/ground_truth.py` uses each tetrahedron's own gradient, conductivity and volume. Lead fields average the same finite-electrode sample positions as openCARP recovery. A sparse map converts nodal Vm into traces from concentric full-thickness rings. Filtering nodal Vm first is valid because these fixed linear spatial operations commute with filtering.

The primary radius is where reconstruction residual energy falls below 20% and stays below it for every larger radius. Membership uses element centroids at 0.5 mm radial bins. The old node-energy radius is removed rather than relabelled as an element quantity.

Raw reconstructions and ring curves remain available for diagnosis. Reported radii require valid physiology, a non-negligible signal, waveform agreement, and the common signed scale from intact unipoles. Crossing the outer boundary is flagged; mesh/domain convergence remains future work.


In [ ]:
contributionTables = []
for condition, measurements in zip(["intact", "lesion"], measurementTables):
    contributions = computeContributions(
        runFolder, condition, config, measurements, gain, physiologyPassed
    )
    contributionTables.append(contributions)
    print(condition, "contribution rows:", len(contributions))


## 10. Apply acceptance and run the clinical-style change-point analysis

`saveExperimentResults` writes the physical pilot decision and automatically runs `boundary_clinical.py`. Far-region profiles are assessed separately for every geometry and gap, with insufficient positions reported explicitly.

The clinical analysis uses log voltage on the lesion side, the 0.01 mV detection limit, 99.9% trimming and an untrimmed sensitivity, and compares exponential versus power-law decay by AIC. It then fits a continuous decay-to-flat-plateau model. Intact profiles are negative controls. The clean no-detection-cut analysis helps distinguish geometry from a measurement floor. Each noise realisation is fitted separately.

With only one slab, Python uses ordinary least squares rather than a patient random effect. Its profile-fit change point is descriptive, not a Bayesian posterior. The optional R companion below runs the original `mcp` model with the clinical chain/iteration settings on these prepared data.


In [ ]:
measurements, pilotPassed = saveExperimentResults(
    runFolder, config, physiologyPassed, measurementTables, contributionTables
)
makeFigures(runFolder, config)

display(json.loads((runFolder / "results" / "pilotStatus.json").read_text()))
display(pd.read_csv(runFolder / "results" / "boundaryChecks.csv"))
for figureName in ["02_clean_voltage_distance.png", "03_contribution_radii.png",
                   "04_example_traces.png", "05_reconstruction_checks.png"]:
    display(Image(filename=str(runFolder / "figures" / figureName)))

clinicalFolder = runFolder / "results" / "clinical"
changePoints = pd.read_csv(clinicalFolder / "changePointFits.csv")
display(changePoints[(changePoints.noiseRepeat == 0)
                     & (changePoints.preprocessing == "clinicalTrimmed")
                     & (changePoints.distanceConvention == "recordingAnchor")])
if (clinicalFolder / "changePointFits.png").exists():
    display(Image(filename=str(clinicalFolder / "changePointFits.png")))


## 11. Interpret and investigate the change point

The anatomical boundary is known at zero. The fitted change point is the **lesion-side distance where decay approaches a plateau**. The contribution radius describes a different quantity: tissue needed to reconstruct a waveform. Do not equate the two or call their difference an error in locating the boundary.

Read `results/clinical/method.json`, `clinicalSampleFlow.csv`, `changePointModelSelection.csv`, `changePointFits.csv`, and `changePointNoiseSummary.csv`. A no-change-point or search-edge result is legitimate. A flat intact control should not be promoted into evidence of a sensing boundary. `changePointAndContributionComparison.csv` joins both quantities at matching recording geometries without asserting equality.

For the original Bayesian model, install R packages `mcp` and `ggdist` plus JAGS on the analysis computer, then run from the project parent:

```bash
Rscript 3D_Code/pilot_change_points.R 3D_Code/runs/boundaryPilotV2
```

The R companion uses the prepared clean clinical-trimmed profiles. Inspect sampler diagnostics and the Python no-change comparison before interpreting posterior estimates. There are no synthetic patient IDs and no peak-frequency claims. The full clinical Rmd remains available for actual multi-patient data.

If physiology fails, the normal notebook stops above. To export explicitly diagnostic results, use the terminal runner with `--process-only --diagnostic` and the same saved simulation configuration. Scientific parameters and acceptance limits must not be changed merely to obtain a pass.

`BOUNDARY_PILOT.md` records the design decisions, expected outputs, resource needs, and next-run instructions.
